# Italy Tripartite School System - Exploratory Analysis

This notebook explores the Italian tripartite school system (primary, lower secondary, upper secondary) using available local datasets. Goals:
- Load and harmonize enrollment and expenditure datasets
- Compare outcomes (graduation rates / NEET proxies) by school level and region
- Visualize spending per student across levels and regions
- Produce summary tables for policy notes

In [ ]:
# Basic environment and helper imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

ROOT = Path('..').resolve() / '.'  # project root when running in notebook folder
DATA_DIR = ROOT / 'local_data'
print('Data dir:', DATA_DIR)

## Load candidate datasets
We'll attempt to load datasets that typically contain school-level or level-disaggregated data: MinIstruzione Alunni, SIOPE processed summary (if present), NEET by education, ISTAT/Eurostat indicators. We'll fall back gracefully when files are missing.

In [ ]:
def load_if_exists(path):
    path = Path(path)
    if path.exists():
        try:
            if path.suffix.lower() == '.csv':
                return pd.read_csv(path)
            elif path.suffix.lower() in ['.xls', '.xlsx']:
                return pd.read_excel(path)
        except Exception as e:
            print('Failed to load', path, e)
    return None

# Candidate files
min_alunni = load_if_exists(DATA_DIR / 'MinIstruzione' / 'Alunni' / 'ALUSECGRADOINDPAR20242520250831.csv')
siope = load_if_exists(DATA_DIR / 'processed' / 'siope_school_expenditure_summary.csv')
neet_by_educ = load_if_exists(DATA_DIR / 'NEET  (giovani non occupati e non in istruzione e formazione) - Dati regionali (IT1,172_931_DF_DCCV_NEET1_6,1.0).csv')
print('Loaded SIOPE?', siope is not None, 'Min Alunni?', min_alunni is not None, 'NEET?', neet_by_educ is not None)

## Harmonization
We'll standardize columns like `year`, `region`, `school_level` and compute derived metrics such as spending per student.

In [ ]:
# Example harmonization when SIOPE + Min Alunni are available
if siope is not None:
    df = siope.copy()
    # standardize columns if present
    if 'anno' in df.columns:
        df['year'] = pd.to_numeric(df['anno'], errors='coerce').astype('Int64')
    if 'importo_euro' in df.columns:
        df['expenditure_euro'] = pd.to_numeric(df['importo_euro'], errors='coerce')
    # prefer denominazione / codice_ente as school identifier
    if 'denominazione' in df.columns:
        df['school_name'] = df['denominazione']
    if 'codice_regione' in df.columns:
        df['region_code'] = df['codice_regione']
    print('SIOPE harmonised rows:', len(df))

if min_alunni is not None:
    al = min_alunni.copy()
    print('Min Alunni columns:', list(al.columns)[:10])

# Merge example (may need tailoring to exact column names)
if siope is not None and min_alunni is not None:
    merged = df.merge(al, left_on=['school_name', 'year'], right_on=['Istituto', 'Anno'], how='left')
    print('Merged shape:', merged.shape)

## Analysis ideas (to implement)
- Spending per student by level and region
- Enrollment trends across levels
- Correlate graduation rate / NEET proxies with spending
- Top/bottom performers by expenditure efficiency

We'll implement these next cells after inspecting available columns and deciding on a merge strategy.